# c05：入库流程、Elasticsearch 验证与 chunk_version 替换策略（教学向）

本笔记本用**短样例**演示 `chunk_version`、ES `count` 校验、以及「注入旧版本 → `delete_chunks_not_matching_version`」；**不包含**真实 OSS 下载与生产入库。

**阿里云 OSS → 本地 → ES 的完整流程**（含入库前 ES 范围预览、按 `source_path` 精确删除防误删）见 **[c06_oss_ingest.ipynb](c06_oss_ingest.ipynb)**。

其余说明见 [ingest_plan_runbook.md](ingest_plan_runbook.md)。**前置**：`uv sync --extra embedding`、Elasticsearch、`.env`。**索引隔离**：默认 `ES_INDEX + "_c05"`。


In [2]:
from __future__ import annotations

import os
import sys
from pathlib import Path

_ROOT = Path.cwd().resolve()
if _ROOT.name == "notebooks":
    _ROOT = _ROOT.parent
_SRC = _ROOT / "src"
if _SRC.is_dir() and str(_SRC) not in sys.path:
    sys.path.insert(0, str(_SRC))

os.chdir(_ROOT)
print("仓库根目录:", _ROOT)
print("已加入 sys.path:", _SRC)


仓库根目录: /Users/zheng/projects/python/ai/rag/rag_law
已加入 sys.path: /Users/zheng/projects/python/ai/rag/rag_law/src


## 0. 依赖检查（torch / FlagEmbedding / elasticsearch）


In [3]:
import importlib.util

print("当前 Python:", __import__("sys").executable)

if importlib.util.find_spec("torch") is None:
    raise RuntimeError("未安装 torch。请执行: uv sync --extra embedding")

import torch

try:
    from FlagEmbedding import BGEM3FlagModel  # noqa: F401
except ModuleNotFoundError as e:
    raise RuntimeError("未安装 FlagEmbedding。请执行: uv sync --extra embedding") from e

try:
    import elasticsearch  # noqa: F401
except ModuleNotFoundError as e:
    raise RuntimeError("未安装 elasticsearch 客户端。请执行: uv sync（dev 组已包含）") from e

print("torch:", torch.__version__, "| elasticsearch OK")


当前 Python: /Users/zheng/projects/python/ai/rag/rag_law/.venv/bin/python


/Users/zheng/projects/python/ai/rag/rag_law/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


torch: 2.11.0 | elasticsearch OK


## 1. 配置、chunk_version 与笔记本专用索引

- **CHUNK_VERSION**：示例 `1.1.7`，对应环境变量 `CHUNK_VERSION` 与 `Settings.chunk_version`。
- **NOTEBOOK_ES_INDEX**：`settings.es_index + "_c05"`。
- **RECREATE_INDEX**：`True` 时先删后建该索引。


In [4]:
from conf.settings import get_settings

get_settings.cache_clear()
settings = get_settings()

CHUNK_VERSION = "1.1.7"
NOTEBOOK_ES_INDEX = settings.es_index + "_c05"
RECREATE_INDEX = True
DELETE_INDEX_AT_END = False

print("es_url:", settings.es_url)
print("NOTEBOOK_ES_INDEX:", NOTEBOOK_ES_INDEX)
print("CHUNK_VERSION:", CHUNK_VERSION)


es_url: http://localhost:9200
NOTEBOOK_ES_INDEX: rag_law_chunks_c05
CHUNK_VERSION: 1.1.7


## 2. 背景：_id、残留与版本策略

- 文档 `_id` = `source_file` + `:` + `chunk_index`。
- 仅覆盖同名 `_id`；块数变少时更高 `chunk_index` 会残留。
- 写入 `chunk_version` 后调用 `delete_chunks_not_matching_version`，删除不等于当前版本的文档（含缺失字段的旧数据）。顺序：先 bulk 新数据，`refresh`，再删除。


## 3. 验证：Elasticsearch 连通性


In [5]:
from es_store.client import elasticsearch_client

client = elasticsearch_client(settings)
assert client.ping(), "ES ping 失败"
print("验证通过: ping OK")


验证通过: ping OK


## 4. 短样例文本并切分


In [6]:
from chunking.split import TextChunk, iter_chunks_for_text

SAMPLE_TEXT = "\n".join(
    [
        "",
        "中华人民共和国宪法（节选样式）",
        "",
        "第一条 中华人民共和国是工人阶级领导的、以工农联盟为基础的人民民主专政的社会主义国家。",
        "",
        "第二条 中华人民共和国的一切权力属于人民。",
    ]
).strip()

CHUNK_SIZE = 300
CHUNK_OVERLAP = 40

chunks: list[TextChunk] = list(
    iter_chunks_for_text(
        SAMPLE_TEXT,
        source_file="c05_notebook_sample.md",
        source_path="notebooks/c05_ingest_es_runbook.ipynb",
        chunk_size=CHUNK_SIZE,
        chunk_overlap=CHUNK_OVERLAP,
        boundary_aware=False,
    )
)
print("块数:", len(chunks))
assert len(chunks) >= 1
for c in chunks[:3]:
    print(f"  [#{c.chunk_index}] len={len(c.text)}")


块数: 1
  [#0] len=83


In [7]:
chunks[0].__dict__

{'text': '中华人民共和国宪法（节选样式）\n\n第一条 中华人民共和国是工人阶级领导的、以工农联盟为基础的人民民主专政的社会主义国家。\n\n第二条 中华人民共和国的一切权力属于人民。',
 'source_file': 'c05_notebook_sample.md',
 'chunk_index': 0,
 'char_start': 0,
 'char_end': 83,
 'source_path': 'notebooks/c05_ingest_es_runbook.ipynb',
 'source_id': 'notebooks/c05_ingest_es_runbook.ipynb',
 'mime_type': 'text/markdown',
 'doc_type': 'law_md',
 'domain': 'law',
 'extra': None}

## 5. 向量化


In [8]:
from embeddings import build_embedder

embedder = build_embedder(settings)
embeddings = embedder.embed_documents([c.text for c in chunks])
dim = embedder.dense_dimension
print("dense_dimension:", dim)
assert len(embeddings) == len(chunks)
print("验证通过: 向量维度")


pre tokenize: 100%|██████████| 1/1 [00:00<00:00, 10.93it/s]
You're using a XLMRobertaTokenizerFast tokenizer. Please note that with a fast tokenizer, using the `__call__` method is faster than using a method to encode the text followed by a call to the `pad` method to get a padded encoding.
Inference Embeddings: 100%|██████████| 1/1 [00:01<00:00,  1.22s/it]

dense_dimension: 1024
验证通过: 向量维度


## 6. 组装 _source（含 chunk_version）


In [9]:
from ingest.documents import chunk_embedding_to_source

sha = "00" * 32

docs = [
    chunk_embedding_to_source(
        c,
        emb,
        source_sha256=sha,
        chunk_version=CHUNK_VERSION,
    )
    for c, emb in zip(chunks, embeddings)
]
assert all(d.get("chunk_version") == CHUNK_VERSION for d in docs)
print("待写入条数:", len(docs), "| chunk_version OK")
docs

待写入条数: 1 | chunk_version OK


[{'text': '中华人民共和国宪法（节选样式）\n\n第一条 中华人民共和国是工人阶级领导的、以工农联盟为基础的人民民主专政的社会主义国家。\n\n第二条 中华人民共和国的一切权力属于人民。',
  'embedding': [0.018111459063151363,
   0.022123611470197882,
   -0.0322798813150618,
   -0.01650787151421655,
   -0.012033674271424882,
   -0.04300420182781678,
   0.04235641007650115,
   0.01517129794641244,
   0.00859184447180012,
   0.020148452557542464,
   -0.04824662123809137,
   0.003541795138975032,
   0.005050580313522021,
   -0.027272545385202487,
   0.012151811602812488,
   0.04486382309997467,
   0.04090500488456867,
   0.004276785785561431,
   -0.02025810242093151,
   -0.010915669744572547,
   -0.026694899473212835,
   -0.005389363031838907,
   -0.034951355626527465,
   0.04486518579379866,
   -0.032331440480522965,
   0.017957777742942646,
   0.011654508239107637,
   0.029475318806434447,
   -0.022067104316472366,
   0.0036339534761866917,
   -0.024505259601077613,
   -0.005353092468767679,
   0.029106869891148468,
   -0.011850771391843951,
   -0.0440922329642216,
   -0.0

## 7. 建索引并 bulk


In [10]:
from es_store.store import EsChunkStore

store = EsChunkStore(client, NOTEBOOK_ES_INDEX, dense_dims=dim)
store.ensure_index(recreate=RECREATE_INDEX)
ok_n, errs = store.bulk_index_chunks(docs)
assert not errs and ok_n == len(docs)
store.refresh()
print("验证通过: bulk OK")


验证通过: bulk OK


## 8. 验证：count 与 chunk_version


In [11]:
total = int(client.count(index=NOTEBOOK_ES_INDEX)["count"])
assert total == len(docs)
r = client.search(index=NOTEBOOK_ES_INDEX, size=10, query={"match_all": {}})
for h in r["hits"]["hits"]:
    assert h["_source"].get("chunk_version") == CHUNK_VERSION
print("验证通过: count=", total, "chunk_version 均为", CHUNK_VERSION)


验证通过: count= 1 chunk_version 均为 1.1.7


## 9. 注入旧版本并验证 delete_chunks_not_matching_version


In [12]:
from chunking.split import TextChunk
from ingest.documents import chunk_embedding_to_source

stale_chunk = TextChunk(
    text="模拟旧版本残留",
    source_file="stale_legacy.md",
    chunk_index=0,
    char_start=0,
    char_end=8,
    source_path="notebooks/stale.md",
)
stale_emb = embedder.embed_documents([stale_chunk.text])[0]
stale_doc = chunk_embedding_to_source(
    stale_chunk,
    stale_emb,
    source_sha256=sha,
    chunk_version="legacy-old",
)
assert not store.bulk_index_chunks([stale_doc])[1]
client.indices.refresh(index=NOTEBOOK_ES_INDEX)
assert int(client.count(index=NOTEBOOK_ES_INDEX)["count"]) == len(docs) + 1

from es_store.version_cleanup import delete_chunks_not_matching_version

resp = delete_chunks_not_matching_version(client, NOTEBOOK_ES_INDEX, CHUNK_VERSION)
print("deleted:", resp.get("deleted"))

assert int(client.count(index=NOTEBOOK_ES_INDEX)["count"]) == len(docs)
for h in client.search(index=NOTEBOOK_ES_INDEX, size=20, query={"match_all": {}})["hits"]["hits"]:
    assert h["_source"].get("chunk_version") == CHUNK_VERSION
print("验证通过: 仅保留当前版本")


Inference Embeddings: 100%|██████████| 1/1 [00:00<00:00,  1.38it/s]


deleted: 1
验证通过: 仅保留当前版本


## 10. 与 rag_ingest / ingest_oss_md_to_es 的关系

设置环境变量 `CHUNK_VERSION` 后，两脚本写入的每条 chunk 均带 `chunk_version`。不删索引时可在 bulk + refresh 后调用 `delete_chunks_not_matching_version`。


## 11. （可选）删除笔记本索引


In [13]:
DELETE_INDEX_AT_END = 1
if DELETE_INDEX_AT_END and client.indices.exists(index=NOTEBOOK_ES_INDEX):
    client.indices.delete(index=NOTEBOOK_ES_INDEX)
    print("已删除:", NOTEBOOK_ES_INDEX)
else:
    print("保留索引:", NOTEBOOK_ES_INDEX)


已删除: rag_law_chunks_c05
